# Machine Learning

This notebook is an introduction to machine learning. We begin by discussing why machine learning matters in accounting, what it means for a machine to "learn," and how supervised and unsupervised learning differ. Then we put those concepts to work. First, we walk through a confusion matrix exercise. Next, we apply these ideas to logistic regression and gradient boosting predictions. Finally, we perform k-means clustering on a sample of recent condo and apartment sales around campus. The purpose is to familiarize you with supervised and unsupervised machine learning basics.

> **A note on scope.** Machine learning is a huge and evolving field, and this lesson is a short, practical introduction. The goal is to understand what "training," "testing," and "classification" mean, to evaluate a classifier with a confusion matrix, and to see two supervised and one unsupervised techniques at work.

Before starting, work through the [Regression](../06_regression/regression.ipynb) notebook. This module assumes you can estimate a regression (including a logistic regression) with `statsmodels` and read the summary output it produces.

## General housekeeping items

As in the previous notebooks, select your kernel before running anything. Click **Select Kernel** in the top right of this notebook, choose **Python Environments...**, and select the `.venv` environment you created in [Getting Started](../01_getting_started/course_intro.ipynb).

Let's begin by importing the necessary libraries. Most are the same packages you have used in earlier modules. The new arrivals are `scikit-learn` (imported as `sklearn`), the most widely used machine learning library in Python, and `xgboost`, a popular gradient boosting library.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score,
                             recall_score)
from sklearn.cluster import KMeans
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

## Why is machine learning important in accounting?

* Working with and analyzing data requires skill sets that are in short supply.
    * A lot of, but not all, accounting work is structured and rules based.
* Machine learning algorithms can automate or assist in these kinds of tasks.
    * It is likely that the accountant of the future will be assisted by, or work alongside, these tools. In fact, this is already happening.

## Why is machine learning such a hot topic?

* Companies have more data than they know what to do with!
* Data analysis techniques are becoming easier to use and more people are learning how to use them.
* Machine learning allows us to scale and automate tasks that traditionally could only be performed by humans (much like all innovation!).

How much data? For a sense of scale, skim LocaliQ's [What Happens in an Internet Minute](https://localiq.com/blog/what-happens-in-an-internet-minute/). In a single minute, people run over 9 million Google searches, send more than 18 million texts, and watch hundreds of millions of short videos, and every one of those actions leaves data behind.

For an example closer to accounting, look at Visa's quarterly [operational performance data](https://s1.q4cdn.com/050606653/files/doc_financials/2026/q3/Q3FY26-Visa-Operational-Performance-Data.pdf): in the 12 months ended June 30, 2026, Visa processed about 275 billion transactions, roughly 750 million per day, and each one is a record that has to be authorized, cleared, settled, and screened for fraud.

### Concepts to remember

* There are significant limitations in what we can learn from data (even "big" data).
    * The complexity of the task is positively related to the amount and quality of data needed to "train" a machine learning algorithm.
    * The more complex the task → the more learning is needed
    * Corporate data is often unstructured, incomplete, or inaccurate. Garbage in, garbage out!

## What is (machine) "learning"?

**Merriam-Webster Dictionary** [definition of learn](https://www.merriam-webster.com/dictionary/learn):

* to gain knowledge or understanding of or skill in by study, instruction, or experience
* to come to be able
* to come to realize

Other definitions from the web:

* Coursera: the science of getting computers to act **without being explicitly programmed.**
* Wikipedia: is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus **perform tasks without explicit instructions.**

An important feature of machine learning is that it generally improves as it gets more and/or better data… similar to how people learn! Think about some of the settings we have explored earlier in the course… Have we "trained" (or taught) an algorithm?

## Types of machine learning

* ML techniques are frequently used for:
    * Prediction, classification, clustering (or "grouping")
* Types of machine learning:
    * Supervised learning
    * Unsupervised learning
    * Reinforcement learning
* We will focus on supervised and unsupervised learning.

## Supervised learning

* Supervised learning techniques leverage data that has been "labeled" by humans or other processes. These techniques are used to develop predictions or classifications.
    * "Labeled" means that we have information on something we want to predict (or classify): animal pictures – "dog or cat", diamonds – "prices", football games – "win or loss", companies – "bankrupt or non-bankrupt", etc.
    * We can think of regression as a supervised learning technique because we have something (y) that we want to predict using inputs (x).
* If all goes well… these predictions or classifications can be applied (or "extrapolated") to unlabeled data… In this sense, the machine "learned" how to predict or classify new, unlabeled data.

The diagram below illustrates the supervised learning workflow. Notice that it includes the training/testing split that we will use later in this notebook.

![](https://miro.medium.com/v2/resize:fit:4800/format:webp/1*589X2eXJJkatGRG-z-s_oA.png)

* Techniques: Regression and classification
* Examples:
    * Spam/not spam (the "classic" example) – ML algorithms can determine the features that identify spam emails using "labeled" data. If the machine "learns" well enough, it can be used to filter your inbox!
    * Cancer/not cancer – ML algorithms can "learn" from images (e.g., MRIs) that have been labeled by doctors to identify tumors in future (unlabeled) images (watch out radiologists!).
    * Fraud/not fraud (credit cards) – Using previous transactions that are classified as fraudulent or legitimate, ML algorithms can identify patterns that can quickly identify fraudulent credit card activity (maybe even send a warning message or turn off the card).

### How do we know if our model is any good?

* Supervised learning requires that we "train" or "teach" our model. After training, we want to see how well it works.
* If we have a large dataset, we might withhold a portion of the observations for testing (we often call this the "test" dataset).
    * We want to see how a model performs using data that it was not trained on… This helps us evaluate how well the model generalizes outside of the data it was trained on.
* For supervised learning techniques we can use predictive power (e.g., R-squared) or confusion matrices to evaluate effectiveness.

## Introduction to the confusion matrix

The confusion matrix is a table that is often used to evaluate the performance of a classification model. Each prediction lands in one of four cells, depending on what the model predicted and what was actually true:

| Actual | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | True Negative (TN) | False Positive (FP) |
| **Actual Positive** | False Negative (FN) | True Positive (TP) |

In this format, the rows hold the actual classes and columns hold the predicted classes, with the negative class listed first. This is the layout that pandas and scikit-learn produce, and it is the one we use throughout this notebook. Be careful: these can follow different orders, so always read the row and column labels rather than relying on position! Which class counts as "positive" is also a choice: it is usually the outcome we are trying to detect (the spam email, the fraudulent transaction, the default), and we will point it out in each example below.

From these four counts we can calculate several common performance metrics:

* **Accuracy** = (True Pos. + True Neg.) / All Instances
    * Overall accuracy of the model. It tells us how many predictions are "true" out of all observations.
* **Precision** = True Pos. / (True Pos. + False Pos.)
    * Accuracy of the model when it predicts a positive. It tells us the percentage of predicted positives that are actual positives.
    * We might focus on precision if the consequences of false positives are very high.
* **Recall** = True Pos. / (True Pos. + False Neg.)
    * Accuracy of the model for actual positives. It tells us the percentage of actual positives that are predicted positives.
    * We might focus on recall if the consequences of false negatives are very high.
* **F1 Score** = 2 * (Precision * Recall) / (Precision + Recall)
    * A harmonic mean of precision and recall. It combines precision and recall, penalizing both false positives and false negatives.
    * We might focus on the F1 score when false positives and false negatives both matter, or when one class is rare and overall accuracy can look strong even though the model handles the rare class poorly.

### Confusion matrix example: spam detection

A confusion matrix is a table used to describe the performance of classifications (or predictions) for which the true values are known. Let's start with the "classic" example from above: email spam detection. We will import the data and inspect the first few rows.

In [ ]:
spam_cm = pd.read_csv('spam_cm.csv')

spam_cm.head()

Each row is one email. The `actual` column records whether the email really was spam, and the `predicted` column records whether the spam filter flagged it as spam. In this example, spam (`True`) is the positive class: a positive prediction means the filter flags an email as spam. Next, let's tabulate the actual labels against the predicted labels and display the counts as a heatmap.

In [ ]:
cm_spam = pd.crosstab(spam_cm['actual'], spam_cm['predicted'],
                      rownames=['Actual'], colnames=['Predicted'])

sns.heatmap(data=cm_spam,
            annot=True,
            annot_kws={'size': 14},
            fmt='g',
            cmap='Blues',
            cbar=False,
            xticklabels=['Not Spam', 'Spam'],
            yticklabels=['Not Spam', 'Spam']
            )

plt.title('Confusion Matrix: Email Spam Detection')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

#### Confusion matrix counts

| Actual | Predicted: Not Spam | Predicted: Spam |
|---|---|---|
| **Actual: Not Spam** | TN = 538 | FP = 20 |
| **Actual: Spam** | FN = 43 | TP = 320 |

* **True Negatives (TN) = 538**: correctly flagged as not spam
* **False Positives (FP) = 20**: flagged spam, was not spam (Type I error)
* **False Negatives (FN) = 43**: missed spam, was actually spam (Type II error)
* **True Positives (TP) = 320**: correctly flagged as spam

This table matches the heatmap above cell for cell: rows are the actual labels and columns are the predicted labels, with the negative class ("Not Spam") listed first, just like the layout in the introduction. Because spam is the positive class, the true positives sit in the bottom-right cell.

#### Classification metrics

**Accuracy** is the overall share of correct predictions:

$$\text{Accuracy} = \frac{TP + TN}{n} = \frac{320 + 538}{921} = 0.932$$

**Precision** tells us how many of the emails flagged as spam were actually spam:

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{320}{320 + 20} = 0.941$$

**Recall** tells us how many of the actual spam emails were caught:

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{320}{320 + 43} = 0.882$$

**F1 Score** is the harmonic mean of precision and recall; it is useful when you want to balance the two rather than optimize one at the expense of the other:

$$\text{F1} = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \times 0.9412 \times 0.8815}{0.9412 + 0.8815} = 0.910$$

We calculated these metrics by hand, but scikit-learn's `classification_report` function produces all of them at once. Each class gets its own row: the `Spam` row treats spam as the positive class and matches our hand calculations above, while the `Not Spam` row reports the same metrics treating "not spam" as the positive class. The `support` column counts the actual observations in each class.

In [ ]:
spam_report = classification_report(spam_cm['actual'],
                                    spam_cm['predicted'],
                                    target_names=['Not Spam', 'Spam'])
print(f"Classification Report:\n{spam_report}")

### Let's apply!

You hear an economist mention that they have perfectly predicted all the economic recessions since 1980. Armed with your new knowledge of confusion matrices, you have decided to comprehensively evaluate the economist's predictions. You collected the data below. Can you create a confusion matrix based on this? Calculate accuracy, precision, and recall. Which metric is the economist referring to?

In [ ]:
cm_example = pd.read_excel('cm_example.xlsx')

cm_example.head()

Next, we will create a confusion matrix to evaluate the accuracy of the economist's predictions. In this example, `Recession` is the positive class: a positive prediction means the economist called a recession. As in the spam example, we tabulate the actual outcomes against the predictions with `pd.crosstab` and display the counts as a heatmap. 

In [ ]:
cm_recession = pd.crosstab(cm_example['recession'],
                           cm_example['prediction'],
                           rownames=['Actual'], colnames=['Predicted'])

sns.heatmap(data=cm_recession,
            annot=True,
            annot_kws={'size': 14},
            fmt='g',
            cmap='Blues',
            cbar=False
            )

plt.title('Confusion Matrix: Recession Predictions')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

As in the spam example, we can generate the full classification report using `scikit-learn`. Look at the `Recession` row: which metric is the economist referring to when they claim to have predicted every recession since 1980?

In [ ]:
recession_report = classification_report(cm_example['recession'],
                                          cm_example['prediction'])
print(f"Classification Report:\n{recession_report}")

## Exercise using logistic regression and default data

Logistic regression can be thought of as a machine learning method that can be used to predict binary outcomes. Let's try using logistic regression to predict defaults using a dataset from the [University of California - Irvine Machine Learning Repository](https://archive.ics.uci.edu/) called "[default of credit card clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)." The file is already included in this folder. Let's import it.

In [ ]:
cc_defaults = pd.read_csv('cc_defaults.csv')

cc_defaults.head()

Take a look at the dataset above and inspect the details from the [source](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients). Here we want to predict the column `default_payment_next_month`. Let's use some basic wrangling to create a set of plausible predictors for default.

Note, this is a simple set of predictors for illustration purposes only. Consider creating new variables that might help you predict default! You could do better!

In [ ]:
# Debt to limit ratio
cc_defaults['dtl_ratio'] = cc_defaults['bill_amt1'] / cc_defaults['limit_bal']

# Change in debt to limit ratio
cc_defaults['dtl_ratio_change'] = (
    (cc_defaults['bill_amt1'] - cc_defaults['bill_amt6'])
    / cc_defaults['limit_bal']
)

# Indicator if ever behind on payments in recent months (any pay_0:pay_6 > 0)
pay_cols = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
cc_defaults['behind'] = (cc_defaults[pay_cols] > 0).any(axis=1)

# Average payment status over the past few months
cc_defaults['avg_status'] = cc_defaults[pay_cols].mean(axis=1)

# Create a column called 'default'
cc_defaults['default'] = cc_defaults['default_payment_next_month']

# Select desired columns
cc_defaults = cc_defaults[['default', 'dtl_ratio', 'dtl_ratio_change',
                           'behind', 'avg_status']]

Now, let’s illustrate “training” and “testing” by splitting the sample into training and testing datasets. Here, we are going to randomly keep 70% of the original dataset and store as “train_data”. The remaining data will be stored as “test_data”.

In [ ]:
cc_train_data, cc_test_data = train_test_split(cc_defaults,
                                               train_size=0.7,
                                               random_state=42,
                                               shuffle=True)

Next, estimate a logistic regression on the training dataset and use the estimated model to predict outcomes in the testing dataset. The `logit()` syntax is the same one we used in the [Regression](../06_regression/regression.ipynb) module.

Note: Here, we convert the 'fitted' values to a binary prediction to force the model to predict either a 1 or a 0 (e.g., greater than or equal to 0.5 is predicted to be a 1, and less than 0.5 is predicted to be a 0). However, we can tinker with that threshold to make the predictions more/less conservative, so we store it in a variable called `logit_cutoff` rather than typing 0.5 into the comparison.

In [ ]:
cc_train_logit = smf.logit(
    "default ~ dtl_ratio + dtl_ratio_change + behind + avg_status",
    data=cc_train_data
).fit()

print(cc_train_logit.summary())

logit_cutoff = 0.5
cc_test_data['fitted'] = cc_train_logit.predict(cc_test_data)
cc_test_data['default_pred'] = (
    (cc_test_data['fitted'] >= logit_cutoff).astype(int)
)

Now that we have both actual and predicted outcomes in the same dataset, let’s create a confusion matrix as above. Here, a default (`1`) is the positive class. Because the outcome is coded 0 or 1, we relabel the heatmap categories as "No Default" and "Default" to make it easier to read. Since those labels are assigned by position, we also reindex the table so it always has both a 0 and a 1 column.

In [ ]:
cm_table_logit = pd.crosstab(cc_test_data['default'],
                             cc_test_data['default_pred'],
                             rownames=['Actual'], colnames=['Predicted'])

# Keep both classes in the table even if the model never predicts one of
# them (e.g., at an extreme threshold) so the heatmap labels stay correct
cm_table_logit = cm_table_logit.reindex(index=[0, 1], columns=[0, 1],
                                        fill_value=0)

sns.heatmap(data=cm_table_logit,
            annot=True,
            annot_kws={'size': 14},
            fmt='g',
            cmap='Blues',
            cbar=False,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default']
            )

plt.title('Confusion Matrix: Default Predictions (Logit)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

Can you label the false positives, false negatives, true positives, and true negatives? Next, let's calculate some classification metrics.

In [ ]:
cc_default = cc_test_data['default']
cc_default_pred = cc_test_data['default_pred']

cc_cm = confusion_matrix(cc_default, cc_default_pred)
print(f"Confusion Matrix:\n{cc_cm}")

cc_report = classification_report(cc_default, cc_default_pred,
                                  target_names=['No Default', 'Default'])
print(f"Classification Report:\n{cc_report}")

Can you interpret the values for accuracy, precision, and recall? Tinker with the prediction threshold (`logit_cutoff` above) and see what happens to these values.

> **A caveat on tinkering.** Adjusting the threshold while watching the test-set scores is a fine way to see the tradeoffs, but once you use the test data to make a choice, its scores no longer give an independent estimate of how the model will do on new data. This concept is similar to data  dredging discussed in the last module. In practice, choices like the threshold or model settings are made using a separate validation sample carved out of the training data, or using cross-validation, and the test set is saved for one final evaluation.

## Gradient boosting (using XGBoost)

Let's repeat the analysis using the credit card data above, this time using decision tree algorithms from machine learning. Here we are going to use a popular technique called Extreme Gradient Boosting from the `xgboost` library.

In [ ]:
# Remove the predictions from the test data that we generated above
cc_test_data = cc_test_data[['default', 'dtl_ratio', 'dtl_ratio_change',
                             'behind', 'avg_status']]

# Split predictors and target (the first column is the target and the
# remaining columns are the predictors)
x_train = cc_train_data.iloc[:, 1:]
y_train = cc_train_data.iloc[:, 0]

x_test = cc_test_data.iloc[:, 1:]
y_test = cc_test_data.iloc[:, 0]

# Initialize XGBoost classifier
xgb_model = XGBClassifier(
    booster='gbtree',              # Use decision tree-based boosting
    objective='binary:logistic',   # Binary classification, probability output
    max_depth=6,                   # Maximum depth of each decision tree (controls model complexity)
    learning_rate=0.05,            # Step size shrinkage to prevent overfitting (smaller = slower but safer)
    gamma=0,                       # Minimum loss reduction required to make a further partition (higher = more conservative)
    n_estimators=100,              # Number of boosting rounds (i.e., number of trees)
    random_state=42,               # Set random seed for reproducible results
    n_jobs=-1                      # Use all available CPU cores for faster training
)

# Fit the model to the training data
xgb_model.fit(x_train, y_train)

# Predict probabilities for the test data and apply the prediction cutoff
xgb_cutoff = 0.5
y_proba = xgb_model.predict_proba(x_test)[:, 1]
y_pred = (y_proba >= xgb_cutoff).astype(int)

As with the logistic regression, let's visualize the XGBoost predictions as a confusion matrix heatmap. Default (`1`) is still the positive class, and we use the same "No Default" and "Default" labels.

In [ ]:
cm_table_xgb = pd.crosstab(y_test, y_pred,
                           rownames=['Actual'], colnames=['Predicted'])

# As above, keep both classes in the table so the labels stay correct
cm_table_xgb = cm_table_xgb.reindex(index=[0, 1], columns=[0, 1],
                                    fill_value=0)

sns.heatmap(data=cm_table_xgb,
            annot=True,
            annot_kws={'size': 14},
            fmt='g',
            cmap='Blues',
            cbar=False,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default']
            )

plt.title('Confusion Matrix: Default Predictions (XGBoost)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

Next, let's calculate the classification metrics so we can compare XGBoost to the logistic regression.

In [ ]:
xgb_cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{xgb_cm}")

xgb_report = classification_report(y_test, y_pred, digits=2,
                                   target_names=['No Default', 'Default'])
print(f"Classification Report:\n{xgb_report}")

Is this better? Before answering, compare both models against a naive baseline: a "model" that always predicts the majority class, no default. Because most customers in the test data did not default, that rule scores a high accuracy without learning anything at all. The code below collects accuracy, precision, and recall (with default as the positive class) for the baseline and both models in one table, along with the probability cutoff each model used, since precision and recall depend on it. Precision is undefined for the baseline because it never predicts a default.

In [ ]:
# A naive baseline: always predict the majority class (no default)
baseline_pred = np.zeros_like(y_test)

# y_test holds the same actual outcomes that cc_default_pred was scored on
predictions = {
    'Always predict no default': baseline_pred,
    'Logistic regression': cc_default_pred,
    'XGBoost': y_pred
}

comparison = pd.DataFrame({
    'Cutoff': [np.nan, logit_cutoff, xgb_cutoff],
    'Accuracy': [accuracy_score(y_test, p) for p in predictions.values()],
    'Precision (default)': [precision_score(y_test, p, zero_division=np.nan)
                            for p in predictions.values()],
    'Recall (default)': [recall_score(y_test, p)
                         for p in predictions.values()]
}, index=list(predictions))

comparison.round(3)

Both models beat the baseline on accuracy, but only by a point or two, which puts the "Is this better?" question in perspective. The larger difference is in what each model gets right. XGBoost catches a greater share of the actual defaulters (higher recall), while a somewhat smaller share of the customers it flags turn out to default (lower precision). So whether XGBoost is "better" depends on the metric, and as we discussed in the introduction, the right metric depends on the relative costs of the two mistakes: a missed default versus a customer wrongly flagged as a risk. While outside the scope of this lesson, we can "tune" the model parameters to improve predictions or adjust tradeoffs between different metrics. Tinker with the values above and see what happens! Keep in mind the caveat from the logistic regression section: if you pick settings based on the test-set scores, the test set no longer provides an independent final evaluation.

## Unsupervised learning

Unsupervised learning techniques use data without a “label” or target outcome to predict. These techniques search for patterns or structure in the data.

“Unlabeled” means that we have information about observations, but no predefined answers: animal pictures without “dog or cat” labels, customers without assigned market segments, properties without assigned location groups, etc.

We can think of clustering as an unsupervised learning technique because we use inputs (x) to discover groups, without an outcome (y) telling us which group each observation belongs to.

If all goes well… these patterns reveal meaningful similarities and differences among observations. In this sense, the machine “learned” how to organize the data, and we use our judgment to interpret what those groups mean. 

Example techniques: Clustering (e.g., k-means) and topic modeling (e.g., latent Dirichlet allocation).


### Clustering

* Clustering groups similar observations into clusters. In k-means, \(k\) is the number of clusters we ask the algorithm to identify. Consider the following example illustrating identification of 3 clusters:

![](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif)

*Figure: [K-means convergence](https://commons.wikimedia.org/wiki/File:K-means_convergence.gif), Wikimedia Commons.*

* How many clusters do we use?
    * When using clustering techniques, it is up to you to identify how many clusters to "impose" on the data. To guide us, we can use a "scree" plot to identify an inflection point (or "elbow") at which the benefit of adding more clusters diminishes.
* How do we know if the clusters are any good?
    * Clusters are identified based on patterns in the data. Strictly speaking, there are no "right" or "wrong" classifications as in supervised learning. However, we should ensure that there is adequate similarity within clusters and dissimilarity between clusters.

### Applications

* A financial analyst searches for patterns in text of corporate filings to identify topics. These topics are simply collections of and patterns in words… the topics themselves are not identified or "labeled."
* Consumer behavior – a retailer that tracks web traffic and browsing history of customers might use unsupervised learning techniques to identify groups of consumers based on purchasing patterns. This can assist with targeted advertising (beware!).
* Many more…

### Clustering exercise

In this folder, you will find a dataset containing information on sales of condos, apartments, and townhomes from around the University of Alabama's campus.<sup>[1](#fn1)</sup>  Let's begin by importing and inspecting the dataset:

In [ ]:
campus_condo_data = pd.read_csv('campus_condo_data.csv')

campus_condo_data.head()

Let's estimate a regression predicting the sales price of a house based on size, bedrooms, and age. Can you interpret the output?

In [ ]:
price_fit1 = smf.ols(
    "np.log(sales_price) ~ np.log(square_feet) + bedrooms + np.log(age+1)",
    data=campus_condo_data
).fit()

results_table = summary_col(price_fit1,
                            stars=True,
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

Let's suppose you wanted to add location to your regression. We have some information on latitude and longitude... around Tuscaloosa, we could think of these as simply coordinates on a map. We might try seeing if we can find groups of properties that were sold near each other. This is a great task for an unsupervised learning technique called clustering. First, we will isolate the latitude and longitude variables and pass them to the k-means clustering function. We will start by grouping the data into 2 clusters.

In [ ]:
# Select the lat/long columns
latlong = campus_condo_data[['long', 'lat']].copy()

# Fit k-means clustering with 2 clusters, 25 random starts
kmeans = KMeans(n_clusters=2,
                n_init=25,
                random_state=42
                )

clusters = kmeans.fit(latlong)

# Store the cluster assignments in a new column
latlong['cluster'] = clusters.labels_

Since latitude and longitude are two dimensional, we can easily plot our data to see how the observations are classified.

In [ ]:
sns.scatterplot(data=latlong,
                x='long',
                y='lat',
                hue='cluster',
                style='cluster',
                palette='colorblind'
                )

plt.title('Real Estate Clusters')
plt.xlabel('Dimension 1: Longitude')
plt.ylabel('Dimension 2: Latitude')
plt.legend(title='Cluster')
plt.show()

As you can see, we have discretion over the number of clusters to select. It is useful to investigate how adding clusters helps differentiate groups of similar observations. There are no "hard and fast" rules when determining the "right" number of clusters. Ultimately, it is up to you to determine the number of clusters that work best in your setting. However, it is a useful exercise to evaluate how much each additional cluster improves the classifications. One way that we can do this is to use a "scree" plot that shows how much incremental variation is explained when adding additional clusters.

In [ ]:
# Select the lat/long columns
latlong = campus_condo_data[['long', 'lat']].copy()

# Loop several iterations of k-means, increasing the cluster count each time
k_values = range(1, 12)

wss_values = []
for k in k_values:
    model = KMeans(n_clusters=k,
                   n_init=25,
                   random_state=42
                   )
    model.fit(latlong)
    wss_values.append(model.inertia_)  # inertia_ = total within-cluster SSE

# Generate a scree plot of the results from each loop
sns.lineplot(x=list(k_values),
             y=wss_values,
             marker='o')

plt.title('Scree Plot for K-Means Clustering')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Total within-cluster sum of squares')
plt.show()

As you can see, the improvement in similarity among observations within each group declines as we add clusters. Let's reperform the clustering exercise above, this time using 4 clusters.

In [ ]:
latlong = campus_condo_data[['long', 'lat']].copy()

kmeans = KMeans(n_clusters=4,
                n_init=25,
                random_state=42
                )

clusters = kmeans.fit(latlong)

latlong['cluster'] = clusters.labels_

sns.scatterplot(data=latlong,
                x='long',
                y='lat',
                hue='cluster',
                style='cluster',
                palette='colorblind'
                )

plt.title('Real Estate Clusters')
plt.xlabel('Dimension 1: Longitude')
plt.ylabel('Dimension 2: Latitude')
plt.legend(title='Cluster')
plt.show()

Finally, let’s add the clusters to the original dataset. Then, let’s reestimate the regression predicting the sales price of a house based on size, bedrooms, age, and location (cluster). Can you interpret the output?

In [ ]:
campus_condo_data['cluster'] = clusters.labels_

price_fit2 = smf.ols(
    "np.log(sales_price) ~ np.log(square_feet) + bedrooms + np.log(age+1)"
    " + C(cluster)",
    data=campus_condo_data
).fit()

results_table = summary_col([price_fit1, price_fit2],
                            stars=True,
                            info_dict={'N': lambda x: f"{int(x.nobs)}"})

print(results_table)

## Wrapping up

In this notebook you learned what it means to "train" and "test" a supervised learning model, used confusion matrices and the accuracy, precision, recall, and F1 metrics to evaluate classifiers, predicted credit card defaults with both logistic regression and gradient boosting, and used k-means clustering, an unsupervised technique, to find structure in data that was never labeled. These tools, and the judgment to evaluate what they produce, are exactly what the "accountant of the future" we described at the start of this notebook will need.

<a id="fn1"><sup>1</sup></a> The data was collected from Zillow in July 2022. The campus_condo_data data includes apartment, condominium, and townhouse sales from Zillow from November 2019 through May 2022. The data was collected from May 2022-July 2022. The search includes sales in Tuscaloosa inside of 359, Warrior River, McFarland Blvd, and 37th E Street. The data was filtered to remove observations with missing values (e.g., incomplete cases), outliers (e.g., sales price of $100), or unlikely to be individual sales (e.g., multiple units included in same sale). Latitude and longitude were obtained from Google Maps, remaining variables were obtained from Zillow. A big thanks to Kimberly Niven, Sara Buroughs, and Taryn Johnson for collecting this data.